In [1]:
import pandas as pd
import pybedtools
import requests
from tqdm import tqdm
from pathlib import Path

DATA_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data")

organism_code = "mm10"
dataset_name = "mouse_liver"

ground_truth_dir = DATA_DIR / "ground_truth_files"

formatted_path = ground_truth_dir / f"chipatlas_{dataset_name}_hepatocytes.csv"

genome_data_dir = DATA_DIR / "genome_data"
tss_bed_file = genome_data_dir / "genome_annotation" / f"{organism_code}" / "gene_tss.bed"
genome_file = genome_data_dir / "reference_genome" / f"{organism_code}" / f"{organism_code}.chrom.sizes"

assert tss_bed_file.exists(), f"TSS BED file not found at {tss_bed_file}"
assert genome_file.exists(), f"Genome file not found at {genome_file}"

# "Oth.Liv.05.AllAg.Hepatocytes.bed"

In [2]:
def download_chip_atlas_cell_type(
    genome, 
    track_type_class="Oth",
    cell_type_class="ALL", 
    threshold="05", 
    track_type="AllAg", 
    cell_type_specific="AllCell",
    timeout=30,
    ground_truth_dir = Path(DATA_DIR) / "ground_truth_files",
    force_redownload=False,
    ):
    """
    Downloads ChIP-Atlas data for a specific cell type and track type, saving it to a specified directory. 
    
    The file will be saved with a name formatted as `{track_type_class}.{cell_type_class}.{threshold}.{track_type}.{cell_type_specific}.bed`.
    This mirrors the naming convention used by ChIP-Atlas for their assembled data files. The function checks if the file already exists 
    in the specified directory before attempting to download it.
    
    If the file already exists and `force_redownload` is False, it will skip the download.
    
    Parameters
    ----------
    genome : str
        The genome assembly to use (e.g., "hg38").
    track_type_class : str, optional
        The track type class to use in the file name (default is "Oth" for "ChIP:TFs and others").
    cell_type_class : str, optional
        The cell type class to use in the file name (default is "ALL" or "All cell types").
    threshold : str, optional
        The threshold to use in the file name (default is "05" for "50").
    track_type : str, optional
        The track type to use in the file name (default is "AllAg" for "All").
    cell_type_specific : str, optional
        The cell type specific to use in the file name (default is "AllCell" for "All").
    timeout : int, optional
        The timeout for the download request in seconds (default is 30).
    ground_truth_dir : Path, optional
        The directory where the downloaded file will be saved (default is "ground_truth_files" within the DATA_DIR).
    force_redownload : bool, optional
        If True, forces the download even if the file already exists (default is False).
    """
    
    file_string = f"{track_type_class}.{cell_type_class}.{threshold}.{track_type}.{cell_type_specific}.bed"
    
    output_path = ground_truth_dir / file_string
    
    if not output_path.exists() or force_redownload:
    
        url = f"https://chip-atlas.dbcls.jp/data/{genome}/assembled/{file_string}"

        print(f"Downloading ChIP-Atlas data from: {file_string}")

        try:
            with requests.get(url, stream=True, timeout=timeout) as r:
                r.raise_for_status()

                total_size = int(r.headers.get("content-length", 0))
                

                with open(output_path, "wb") as f:
                    with tqdm(
                        total=total_size,
                        unit="B",
                        unit_scale=True,
                        unit_divisor=1024,
                        desc="Downloading",
                    ) as pbar:
                        for chunk in r.iter_content(chunk_size=8192):
                            if chunk:  # skip keep-alive chunks
                                f.write(chunk)
                                pbar.update(len(chunk))

        except Exception as e:
            print(f"Error fetching ChIP-Atlas data: {e}")
    else:
        print(f"File already exists at {output_path}, skipping download.")
        
    return file_string

def map_chip_atlas_peaks_to_closest_tss(
    chip_atlas_bed_file: str,
    tss_bed_file: str,
    genome_file: str
) -> pd.DataFrame:
    """
    Map Chip-Atlas peaks to their closest gene TSS within a specified distance cutoff.
    
    Parameters
    ----------
    chip_atlas_bed_file : str
        Path to the Bed file containing Chip-Atlas peaks.
    tss_bed_file : str
        Path to the Bed file containing gene TSS locations.
    genome_file : str
        Path to the genome file for sorting BedTools objects.
        
    Returns
    -------
    pd.DataFrame : pd.DataFrame
        DataFrame containing peak-gene pairs with their distances.
    """
    chip_bed = pybedtools.BedTool(chip_atlas_bed_file)
    tss_bed = pybedtools.BedTool(tss_bed_file)

    # drop random/chrUn contigs
    tss_bed = tss_bed.filter(lambda f: "random" not in f.chrom and "chrUn" not in f.chrom).saveas()

    # Sort both consistently
    if genome_file:
        chip_sorted = chip_bed.sort(g=genome_file)
        tss_sorted  = tss_bed.sort(g=genome_file)
    else:
        chip_sorted = chip_bed.sort()
        tss_sorted  = tss_bed.sort()

    # Now run closest on the sorted objects
    chip_closest_tss = chip_sorted.closest(tss_sorted, d=True)
    
    raw_chip_closest_tss_df = chip_closest_tss.to_dataframe(
    names=[
            # fields from the ChIP peak AFTER chrom/start/end:
            "peak_name",
            "peak_score",
            "peak_strand",
            "peak_thick_start",
            "peak_thick_end",
            "peak_rgb",
            "tss_chr",
            "tss_start",
            "tss_end",
            "tss_gene",
            "distance"
        ]
    ).reset_index()
    
    raw_chip_closest_tss_df = raw_chip_closest_tss_df.rename(
    columns={
            "level_0": "peak_chr",
            "level_1": "peak_start",
            "level_2": "peak_end",
        }
    )
    raw_chip_closest_tss_df["source_id"] = (
    raw_chip_closest_tss_df["peak_name"]
        .astype(str)
        .str.extract(r"Name=([^%]+)", expand=False)
        .str.upper()
    )

    raw_chip_closest_tss_df["peak_id"] = (
        raw_chip_closest_tss_df["peak_chr"].astype(str)
        + ":" +
        raw_chip_closest_tss_df["peak_start"].astype(str)
        + "-" +
        raw_chip_closest_tss_df["peak_end"].astype(str)
    )
    
    raw_chip_closest_tss_df["source_id"] = raw_chip_closest_tss_df["source_id"].str.upper()
    raw_chip_closest_tss_df["target_id"] = raw_chip_closest_tss_df["tss_gene"].str.upper()

    chip_closest_tss_df = raw_chip_closest_tss_df[["source_id", "peak_id", "target_id", "distance"]]
    
    return chip_closest_tss_df

In [3]:
file_string = download_chip_atlas_cell_type(
    genome=organism_code,
    track_type_class="Oth",
    cell_type_class="Liv",
    threshold="05",
    track_type="AllAg",
    cell_type_specific="Hepatocytes"
)

chip_atlas_bed_file = ground_truth_dir / file_string
print(f"Chip-Atlas bed file path:\n    - {chip_atlas_bed_file.name}")

print("\nMapping Chip-Atlas peaks to closest TSS...")
if not formatted_path.exists():
    chip_atlas_tf_peak_tg_dist = map_chip_atlas_peaks_to_closest_tss(
        chip_atlas_bed_file=str(chip_atlas_bed_file),
        tss_bed_file=str(tss_bed_file),
        genome_file=str(genome_file)
    )
    
    chip_atlas_tf_peak_tg_dist = chip_atlas_tf_peak_tg_dist[["source_id", "target_id"]].drop_duplicates()
    
    num_tfs = chip_atlas_tf_peak_tg_dist["source_id"].nunique()
    num_tgs = chip_atlas_tf_peak_tg_dist["target_id"].nunique()
    num_edges = len(chip_atlas_tf_peak_tg_dist)

    print(f"Number of unique TFs: {num_tfs}")
    print(f"Number of unique TGs: {num_tgs}")
    print(f"Number of TF-TG edges: {num_edges}")

    chip_atlas_tf_peak_tg_dist.to_csv(formatted_path, index=False)
    print(f"Formatted Chip-Atlas peak to gene TSS distances saved to:\n    - {formatted_path}")
else:
    print(f"Formatted file already exists at {formatted_path}, skipping mapping step.")

Downloading: 100%|█████████████████████████████████████████████████████████████████████| 219M/219M [00:14<00:00, 16.1MB/s]


Chip-Atlas bed file path:
    - Oth.Liv.05.AllAg.Hepatocytes.bed

Mapping Chip-Atlas peaks to closest TSS...
Number of unique TFs: 24
Number of unique TGs: 22142
Number of TF-TG edges: 134599
Formatted Chip-Atlas peak to gene TSS distances saved to:
    - /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/ground_truth_files/chipatlas_mouse_liver_hepatocytes.csv
